In [2]:
import sys

import numpy as np
import sklearn
import torch

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("PyTorch:", torch.__version__)

Python: 3.13.18
NumPy: 2.4.6
scikit-learn: 1.6.1
PyTorch: 2.11.0+cpu


In [3]:
from sklearn.datasets import load_digits

X, y = load_digits(return_X_y=True)

print("Размер X:", X.shape)
print("Размер y:", y.shape)
print("Минимальное значение пикселя:", X.min())
print("Максимальное значение пикселя:", X.max())
print("Классы:", np.unique(y))

Размер X: (1797, 64)
Размер y: (1797,)
Минимальное значение пикселя: 0.0
Максимальное значение пикселя: 16.0
Классы: [0 1 2 3 4 5 6 7 8 9]


In [4]:
from sklearn.model_selection import train_test_split

split_seed = 0
all_indices = np.arange(len(y))

rest_indices, test_indices = train_test_split(
    all_indices,
    test_size=0.2,
    stratify=y,
    random_state=split_seed,
)

train_indices, val_indices = train_test_split(
    rest_indices,
    test_size=0.25,
    stratify=y[rest_indices],
    random_state=split_seed,
)

print("train:", len(train_indices))
print("val:", len(val_indices))
print("test:", len(test_indices))

print("Пересечение train и val:", len(set(train_indices) & set(val_indices)))
print("Пересечение train и test:", len(set(train_indices) & set(test_indices)))
print("Пересечение val и test:", len(set(val_indices) & set(test_indices)))

combined = set(train_indices) | set(val_indices) | set(test_indices)
print("Всего уникальных индексов:", len(combined))

for name, indices in [
    ("train", train_indices),
    ("val", val_indices),
    ("test", test_indices),
]:
    print(name, "классы:", np.unique(y[indices]))

train: 1077
val: 360
test: 360
Пересечение train и val: 0
Пересечение train и test: 0
Пересечение val и test: 0
Всего уникальных индексов: 1797
train классы: [0 1 2 3 4 5 6 7 8 9]
val классы: [0 1 2 3 4 5 6 7 8 9]
test классы: [0 1 2 3 4 5 6 7 8 9]


In [5]:
from pathlib import Path
import hashlib
import json

data_directory = Path("data")
data_directory.mkdir(exist_ok=True)

split_data = {
    "dataset": "sklearn.datasets.load_digits",
    "split_seed": split_seed,
    "train": sorted(int(index) for index in train_indices),
    "val": sorted(int(index) for index in val_indices),
    "test": sorted(int(index) for index in test_indices),
}

split_path = data_directory / "split.json"

with split_path.open("w", encoding="utf-8", newline="\n") as file:
    json.dump(split_data, file, ensure_ascii=False, indent=2, sort_keys=True)
    file.write("\n")

split_sha256 = hashlib.sha256(split_path.read_bytes()).hexdigest()

print("Создан файл:", split_path)
print("Размер файла:", split_path.stat().st_size, "байт")
print("SHA-256:", split_sha256)

Создан файл: data/split.json
Размер файла: 16975 байт
SHA-256: d3ba070d3282c94e67fcf0f0bd7c10e0efc97dbcb00c07397313e14d38b5931a


In [6]:
manifest_data = {
    "schema_version": 1,
    "datasets": {
        "load_digits": {
            "source": "sklearn.datasets.load_digits",
            "package": "scikit-learn",
            "package_version": sklearn.__version__,
            "total_records": int(len(y)),
            "features_per_record": int(X.shape[1]),
            "classes": [int(value) for value in np.unique(y)],
            "feature_value_range": [
                float(X.min()),
                float(X.max()),
            ],
            "selection": {
                "included_records": int(len(y)),
                "excluded_records": 0,
                "rule": "Использованы все записи набора load_digits без исключений",
            },
            "split": {
                "file": "data/split.json",
                "sha256": split_sha256,
                "split_seed": split_seed,
                "rule": (
                    "Стратифицированное разбиение: сначала 20% в test, "
                    "затем 25% остатка в val"
                ),
                "sizes": {
                    "train": int(len(train_indices)),
                    "val": int(len(val_indices)),
                    "test": int(len(test_indices)),
                },
            },
        }
    },
}

manifest_path = data_directory / "manifest.json"

with manifest_path.open("w", encoding="utf-8", newline="\n") as file:
    json.dump(manifest_data, file, ensure_ascii=False, indent=2, sort_keys=True)
    file.write("\n")

print("Создан файл:", manifest_path)
print("Размер файла:", manifest_path.stat().st_size, "байт")
print("Версия scikit-learn:", sklearn.__version__)
print("Хеш split.json в манифесте:", manifest_data["datasets"]["load_digits"]["split"]["sha256"])

Создан файл: data/manifest.json
Размер файла: 1075 байт
Версия scikit-learn: 1.6.1
Хеш split.json в манифесте: d3ba070d3282c94e67fcf0f0bd7c10e0efc97dbcb00c07397313e14d38b5931a


In [7]:
from pathlib import Path

print("Текущая папка:", Path.cwd())
print("Папка data существует:", Path("data").exists())

if Path("data").exists():
    print("Файлы внутри data:", [path.name for path in Path("data").iterdir()])

Текущая папка: /content
Папка data существует: True
Файлы внутри data: ['manifest.json', 'split.json']
